# Poem Labeling Tool
Label each poem **immigration** or **not immigration**. Progress saves to `data/df_sample.csv` after each action.

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

CSV_PATH = 'data/df_sample.csv'
df = pd.read_csv(CSV_PATH)

if 'label' not in df.columns:
    df['label'] = None
    df.to_csv(CSV_PATH, index=False)

state = {'idx': None}
history = []  # list of (idx, old_label) for undo

def get_next_unlabeled(after=None):
    unlabeled = df[df['label'].isna()].index.tolist()
    if not unlabeled:
        return None
    if after is not None and after in unlabeled:
        pos = unlabeled.index(after)
        return unlabeled[(pos + 1) % len(unlabeled)]
    return unlabeled[0]

state['idx'] = get_next_unlabeled()

# --- Widgets ---
progress_bar = widgets.IntProgress(
    min=0, max=len(df), value=int(df['label'].notna().sum()),
    layout=widgets.Layout(width='100%'),
    style={'bar_color': '#c4a882'}
)
progress_label = widgets.HTML()
poem_display   = widgets.HTML(layout=widgets.Layout(max_height='620px', overflow_y='auto'))

def styled_btn(label, bg, width='190px'):
    return widgets.Button(
        description=label,
        layout=widgets.Layout(width=width, height='44px'),
        style={'button_color': bg, 'font_weight': 'bold'}
    )

imm_btn     = styled_btn('✈  Immigration  [I]',     '#3a5a7a')
not_imm_btn = styled_btn('✗  Not Immigration  [N]', '#6a4a2a')
skip_btn    = styled_btn('Skip  [S]',                '#7a6a3a', width='110px')
undo_btn    = styled_btn('↩  Undo  [U]',             '#555555', width='110px')

def update_display():
    idx = state['idx']
    labeled = int(df['label'].notna().sum())
    total = len(df)
    remaining = total - labeled
    progress_bar.value = labeled

    if idx is None:
        progress_label.value = f'<b style="color:#3a5a7a;font-family:sans-serif;">All {total} poems labeled!</b>'
        for btn in [imm_btn, not_imm_btn, skip_btn]:
            btn.disabled = True
        poem_display.value = '<h3 style="color:#3a5a7a;font-family:Georgia,serif;padding:20px;">All done!</h3>'
        return

    row = df.iloc[idx]
    existing = row['label'] if pd.notna(row['label']) else None
    badge = ''
    if existing:
        color = '#3a5a7a' if existing == 'immigration' else '#6a4a2a'
        badge = (f' <span style="background:{color};color:#f7f3ed;padding:2px 9px;'
                 f'border-radius:4px;font-size:11px;font-family:sans-serif;'
                 f'letter-spacing:0.5px;text-transform:uppercase;">{existing}</span>')

    try:
        year = int(row['Year'])
    except (ValueError, TypeError):
        year = 'N/A'

    progress_label.value = (
        f'<span style="font-size:13px;font-family:sans-serif;color:#8a7060;">'
        f'<b style="color:#2d2520">{labeled}/{total}</b> labeled &nbsp;&bull;&nbsp; '
        f'<b style="color:#2d2520">{remaining}</b> remaining</span>'
    )

    poem_lines = str(row['Poem Text']).replace('\n', '<br>')
    poem_display.value = f"""
    <div style="font-family:Georgia,serif;max-width:640px;padding:36px 44px;
                background:#fdf8f2;border:1px solid #ddd4c4;border-radius:12px;
                box-shadow:0 3px 14px rgba(120,90,50,0.09);">
      <h2 style="margin:0 0 6px 0;font-size:20px;color:#1e1612;font-weight:normal;
                 letter-spacing:0.5px;font-style:italic;">{row['Title']}{badge}</h2>
      <p style="color:#b09a86;margin:0 0 22px 0;font-size:12px;font-family:sans-serif;
                letter-spacing:0.8px;text-transform:uppercase;">
        {row['Poet']} &nbsp;&middot;&nbsp; {year}
      </p>
      <div style="line-height:1.85;font-size:16px;color:#2d2520;">{poem_lines}</div>
    </div>
    """

def classify(label):
    idx = state['idx']
    if idx is None:
        return
    old = df.at[idx, 'label'] if pd.notna(df.at[idx, 'label']) else None
    history.append((idx, old))
    df.at[idx, 'label'] = label
    df.to_csv(CSV_PATH, index=False)
    state['idx'] = get_next_unlabeled()
    update_display()

def on_skip(b):
    idx = state['idx']
    if idx is None:
        return
    old = df.at[idx, 'label'] if pd.notna(df.at[idx, 'label']) else None
    history.append((idx, old))
    state['idx'] = get_next_unlabeled(after=idx)
    update_display()

def on_undo(b):
    if not history:
        return
    idx, old_label = history.pop()
    df.at[idx, 'label'] = old_label
    df.to_csv(CSV_PATH, index=False)
    state['idx'] = idx
    update_display()

imm_btn.on_click(lambda b: classify('immigration'))
not_imm_btn.on_click(lambda b: classify('not immigration'))
skip_btn.on_click(on_skip)
undo_btn.on_click(on_undo)

button_row = widgets.HBox(
    [imm_btn, not_imm_btn, skip_btn, undo_btn],
    layout=widgets.Layout(gap='10px', padding='10px 0 8px 0')
)

display(HTML("""
<div style="background:#f7f3ed;padding:16px 0 8px 0;font-family:sans-serif;">
  <div style="font-size:20px;font-weight:700;color:#1e1612;letter-spacing:0.4px;">Poem Labeler</div>
  <div style="font-size:12px;color:#a08878;margin-top:5px;">
    Hotkeys:
    &nbsp;<kbd style="background:#e8dfd2;border:1px solid #ccc4b8;border-radius:3px;padding:1px 5px;">I</kbd> Immigration
    &nbsp;<kbd style="background:#e8dfd2;border:1px solid #ccc4b8;border-radius:3px;padding:1px 5px;">N</kbd> Not Immigration
    &nbsp;<kbd style="background:#e8dfd2;border:1px solid #ccc4b8;border-radius:3px;padding:1px 5px;">S</kbd> Skip
    &nbsp;<kbd style="background:#e8dfd2;border:1px solid #ccc4b8;border-radius:3px;padding:1px 5px;">U</kbd> Undo
  </div>
</div>
"""))
display(progress_label)
display(progress_bar)
display(button_row)
display(poem_display)
update_display()

display(HTML("""
<script>
(function() {
  document.removeEventListener('keydown', window._poemHotkey);
  window._poemHotkey = function(e) {
    var tag = document.activeElement.tagName;
    if (tag === 'INPUT' || tag === 'TEXTAREA' || document.activeElement.isContentEditable) return;
    if (e.metaKey || e.ctrlKey || e.altKey) return;
    var key = e.key.toLowerCase();
    var btns = Array.from(document.querySelectorAll('button.widget-button, .jp-Button'));
    if (key === 'i') {
      e.preventDefault();
      btns.find(b => b.textContent.includes('Immigration') && !b.textContent.includes('Not'))?.click();
    } else if (key === 'n') {
      e.preventDefault();
      btns.find(b => b.textContent.includes('Not Immigration'))?.click();
    } else if (key === 's') {
      e.preventDefault();
      btns.find(b => b.textContent.includes('Skip'))?.click();
    } else if (key === 'u') {
      e.preventDefault();
      btns.find(b => b.textContent.includes('Undo'))?.click();
    }
  };
  document.addEventListener('keydown', window._poemHotkey);
})();
</script>
"""))

HTML(value='')

IntProgress(value=0, layout=Layout(width='100%'), max=375, style=ProgressStyle(bar_color='#c4a882'))

HTML(value='', layout=Layout(max_height='620px'))

In [2]:
df_sample = pd.read_csv('data/df_sample.csv')

In [3]:
df_sample_nolables = df_sample.drop(columns=['label'])

In [4]:
df_sample_nolables.head()

,Title,Poet,Year,URL,Poem Text
0,things that shine in the night,Rigoberto González,2016.0,https://poets.org/poem/things-shine-night,Fulgencio’s silver crown—when he snores\nthe m...
1,In a Landscape: IV,John Gallaher,2013.0,https://poets.org/poem/landscape-iv,"Now the scene changes, we say, and the next fe..."
2,Bird,Niki Herd,2020.0,https://poets.org/poem/bird,"Yesterday, at Shepherd and Gray, the parking l..."
3,"The Aeneid, Book VI [First, the sky and the ea...",Virgil,2006.0,https://poets.org/poem/aeneid-book-vi-first-sk...,"“First, the sky and the earth and the flowing ..."
4,"Pandemic: While home is an outbreak, we pass a...",Arianne True,2020.0,https://poets.org/poem/pandemic-while-home-out...,--\n--\n--\n--\n--\n--\nThis country has a way...


In [5]:
df_sample_nolables.to_csv('data/df_sample_nolabels.csv', index=False)